In [ ]:
# -----------------------------
# 1. Imports
# -----------------------------
import fitz  # PyMuPDF
import json
from transformers import T5Tokenizer, T5ForConditionalGeneration
from langchain.text_splitter import CharacterTextSplitter

# ---------------------------
# 2. Load PDF
# -----------------------------
pdf_path = "/content/temppdf.pdf"  # Change this to your PDF path

try:
    doc = fitz.open(pdf_path)
    print(f"✅ Successfully opened: {pdf_path}")
except FileNotFoundError:
    raise FileNotFoundError(f"❌ File not found: {pdf_path}")
except Exception as e:
    raise RuntimeError(f"❌ Error opening PDF: {e}")

# -----------------------------
# 3. Extract Paragraphs with Page Numbers
# -----------------------------
paragraphs_with_page = []

for page_num in range(doc.page_count):
    page = doc[page_num]
    blocks = page.get_text("blocks")  # returns list of text blocks
    for block in blocks:
        if block[6] == 0:  # it's a text block
            text = block[4].strip()
            if len(text) > 30:  # Filter out short blocks/noise
                paragraphs_with_page.append({
                    "text": text,
                    "page_num": page_num + 1  # 1-indexed
                })

print(f"📄 Total extracted paragraphs: {len(paragraphs_with_page)}")

# -----------------------------
# 4. Split Large Paragraphs
# -----------------------------
splitter = CharacterTextSplitter(
    separator=" ",
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
)

final_paragraphs = []

for para in paragraphs_with_page:
    chunks = splitter.split_text(para["text"])
    for chunk in chunks:
        final_paragraphs.append({
            "text": chunk,
            "page_num": para["page_num"]
        })

print(f"✂ Total chunks after splitting: {len(final_paragraphs)}")

# -----------------------------
# 5. Load T5 Model and Tokenizer
# -----------------------------
model_name = "google/flan-t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
print(f"🤖 Loaded model: {model_name}")

# -----------------------------
# 6. Generate Headings
# -----------------------------
def generate_heading(paragraph_text: str) -> str:
    prompt = "Generate a title: " + paragraph_text
    input_ids = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).input_ids
    output_ids = model.generate(
        input_ids,
        max_length=30,
        num_beams=4,
        early_stopping=True
    )
    heading = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return heading

# -----------------------------
# 7. Process All Paragraphs
# -----------------------------
output_data = []

for i, para in enumerate(final_paragraphs):
    print(f"🔄 Generating heading for paragraph {i+1}/{len(final_paragraphs)} "
          f"(Page {para['page_num']})...")
    heading = generate_heading(para["text"])
    output_data.append({
        "page_number": para["page_num"],
        "generated_heading": heading
    })
    print(f"➡ {heading}")
    print("---")

# -----------------------------
# 8. Export Output as JSON
# -----------------------------
json_output = json.dumps(output_data, indent=4)
print("\n📦 Final JSON Output:")
print(json_output)

# Optional: Save to file
with open("headings_output.json", "w", encoding="utf-8") as f:
    f.write(json_output)
print("📁 Output saved to headings_output.json")
